In [ ]:
%reload_ext autoreload
%autoreload 2

import os
import sys

import mne
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd

# Steps up one level and finds the 'src' folder
module_path = os.path.abspath(os.path.join("..", "src"))

if module_path not in sys.path:
    sys.path.append(module_path)

from eeg_pipeline import (
    load_raw,
    preprocess_raw,
    create_epochs,
    compute_psd_per_epoch,
    average_psd_per_event,
    process_single_recording,
    process_subject,
    subject_avg_to_dataframe,
)

SCRIPT FLOW

```python
    subject_avg = process_subject(
        gdf_files  = gdf_files,
        output_dir = OUTPUT_DIR,
        subject_id = SUBJECT_ID,
        condition  = CONDITION,
        # epoch window
        tmin       = -0.5,
        tmax       = 4.0,
        baseline   = None,       # no voltage-level baseline; rely on detrend=1
        # artifact rejection
        reject_uv  = 150.0,
        flat_uv    = 1.0,
        # preprocessing
        l_freq     = 0.5,
        h_freq     = 100.0,
        # PSD — Welch, 2 s segments at 256 Hz → Δf = 0.5 Hz, 50 % overlap
        psd_method = "welch",
        fmin       = 1.0,
        fmax       = 45.0,
        n_per_seg  = 512,
        n_overlap  = 256,
    )
```

In [5]:
SUBJECT_ID = "S03_ME"
CONDITION  = "motor_execution"   # or "motor_imagery" for Step 5
OUTPUT_DIR = "./results"

SUBJECT_NUM = str(int(SUBJECT_ID.split("_")[0][1:]))  # Extract the subject number from the SUBJECT_ID
FILE_NAME  = f"{CONDITION.replace('_', '')}_subject{SUBJECT_NUM}"

gdf_files = [
        f"../data/{CONDITION}/{SUBJECT_ID}/{FILE_NAME}_run{i:d}.gdf"
        for i in range(1, 2)
    ]

print("gdf_files:", gdf_files)

gdf_files: ['../data/motor_execution/S03_ME/motorexecution_subject3_run1.gdf']


In [24]:
raw, events, event_id = load_raw(gdf_files[0])

print("\n", raw, "\n\n", events, "\n\n", event_id)

[load]    motorexecution_subject3_run1.gdf | 64 ch | 168448 samples (329.0 s) | 42 events

 <RawGDF | motorexecution_subject3_run1.gdf, 64 x 168448 (329.0 s), ~52 KiB, data not loaded> 

 [[  3584      0   1536]
 [  7494      0   1536]
 [ 11377      0   1540]
 [ 15274      0   1538]
 [ 19229      0   1536]
 [ 23109      0   1541]
 [ 27168      0   1536]
 [ 31025      0   1542]
 [ 34685      0   1539]
 [ 38516      0   1538]
 [ 42317      0   1540]
 [ 46348      0   1542]
 [ 50236      0   1538]
 [ 53952      0   1542]
 [ 58009      0   1541]
 [ 61827      0   1537]
 [ 65494      0   1539]
 [ 69330      0   1537]
 [ 73157      0   1540]
 [ 77060      0   1540]
 [ 80935      0   1542]
 [ 84715      0   1541]
 [ 88670      0   1539]
 [ 92660      0   1537]
 [ 96574      0   1536]
 [100219      0   1536]
 [103895      0   1541]
 [107805      0   1538]
 [111759      0   1541]
 [115617      0   1538]
 [119614      0   1537]
 [123227      0   1539]
 [127285      0   1539]
 [131283      0   15

In [23]:
%reload_ext autoreload
%autoreload 2

from eeg_pipeline import (
    load_raw,
    preprocess_raw,
    create_epochs,
    compute_psd_per_epoch,
    average_psd_per_event,
    process_single_recording,
    process_subject,
    subject_avg_to_dataframe,
)

In [25]:
raw = preprocess_raw(raw, l_freq=0.5, h_freq=50.0)

print("\n", raw)

Reading 0 ... 168447  =      0.000 ...   328.998 secs...
[preproc] ref=average | bp=0.5–50.0 Hz | notch=None Hz | ch=['C3', 'C1', 'Cz', 'C2', 'C4']

 <RawGDF | motorexecution_subject3_run1.gdf, 5 x 168448 (329.0 s), ~6.4 MiB, data loaded>


In [26]:
epochs = create_epochs(raw, events, tmin=-1.0, tmax=4.0, baseline=(None, 0), reject_uv=None, flat_uv=None, verbose=True)

print("\n", epochs)

Not setting metadata
42 matching events found
Setting baseline interval to [-1.0, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 42 events and 2561 original time points ...


ValueError: array must not contain infs or NaNs